In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('Merged_Data_5.tsv', sep='\t')
df.head()

,ExpNum,X_2,X_1,Experiment,X,Column_array_EcoMAC,CEL_file_name,GEO_accession,Author,Date,...,yjtD,yfjD,ygcU,ygfQ,yggP,yhaM,yhdP,rtcA,yigE,Gene_End
0,1,1,1,1,1,590,PU00249,GSE1121,M._Covert,20040309,...,9.15086,9.31306,8.80685,8.02362,6.89029,8.69035,8.79187,9.17791,8.40815,NaN
1,2,2,2,1,2,591,PU00250,GSE1121,M._Covert,20040309,...,9.09961,9.25085,8.98365,7.84334,6.93732,8.67660,8.60992,9.19737,8.48508,NaN
2,3,3,3,1,3,592,PU00251,GSE1121,M._Covert,20040309,...,9.32948,9.34659,8.40375,7.28272,6.74818,8.10396,8.69566,9.26533,8.70936,NaN
3,4,4,4,2,4,593,PU00252,GSE1121,M._Covert,20040309,...,9.22140,9.37100,9.04991,8.07718,6.92423,9.00301,8.73660,9.32321,8.59597,NaN
4,5,5,5,2,5,594,PU00253,GSE1121,M._Covert,20040309,...,9.20407,9.41722,8.89244,7.88387,7.00594,8.92962,9.01716,9.24857,8.54762,NaN


In [2]:
sig_params_only_df = pd.read_csv('significant-params.tsv', sep='\t')
sig_params_only_df.head(15)

,param,n,pearson,p,sigma
0,ENVIRONMENTAL,7,NaN,NaN,NaN
1,Strain_MG1655,589,-0.3665,3.095000e-18,9.5424
2,Treatment_pH,589,0.2394,5.037000e-07,5.9736
3,Medium_LB_HOMOPIPES,589,0.2439,2.528000e-07,6.0928
4,Treatment_KCl,589,0.2439,2.528000e-07,6.0928
5,Treatment_DMF,589,0.3341,8.371000e-15,8.5889
6,Treatment_biofilm,589,0.3554,5.212000e-17,9.2111
7,Strain_BW25113,589,0.3905,4.400000e-21,10.2775
8,GENE,1265,NaN,NaN,NaN
9,gltI,589,-0.5413,3.533000e-46,15.5989


In [3]:
#take out first 9 rows so only keeps the gene params 

sig_params_only_df = sig_params_only_df.iloc[9:].reset_index(drop=True)
sig_params_only_df.head(100)

,param,n,pearson,p,sigma
0,gltI,589,-0.5413,3.533000e-46,15.5989
1,nlpA,589,-0.5058,6.219000e-39,14.2064
2,purL,589,-0.5050,9.094000e-39,14.1737
3,nadB,589,-0.4843,5.470000e-35,13.4102
4,nadA,589,-0.4792,4.091000e-34,13.2295
...,...,...,...,...,...
95,ilvB,589,-0.3617,1.047000e-17,9.4005
96,yrhB,589,-0.3613,1.165000e-17,9.3881
97,pheP,589,-0.3610,1.254000e-17,9.3795
98,livM,589,-0.3604,1.478000e-17,9.3602


In [4]:
#have only the gene columns in the actual dataset while preserving column order 
df2= sig_params_only_df
cols = df2['param'].tolist()
df1 = df[[c for c in cols if c in df.columns]]
df1.head()

,gltI,nlpA,purL,nadB,nadA,xanP,sstT,pstS,yhjE,argH,...,eco,cspG,add,ycdZ,melA,kbl,tppB,nhaA,udp,cdd
0,10.9783,10.2568,10.0293,9.36703,10.7273,9.89200,9.42279,10.1601,10.68370,10.2826,...,9.50655,9.06924,9.22140,9.29754,8.52657,9.75266,8.99107,9.70365,9.72784,8.85439
1,10.8914,10.2594,10.0001,9.12113,10.6034,9.81119,9.25009,10.0248,10.40790,10.1902,...,9.65040,9.21160,9.29990,9.23802,8.52337,9.62357,8.98660,9.67593,9.78355,8.65507
2,10.9833,10.4555,10.1123,9.25009,10.6819,9.83725,9.42441,10.0966,10.53380,10.4048,...,9.25085,9.08107,9.25770,9.37178,8.48751,9.68973,9.06261,9.78840,9.53692,8.63255
3,10.2717,10.2514,10.1687,9.36543,10.9304,9.97354,10.58260,10.1464,9.85967,10.6121,...,9.61472,8.89169,9.24101,9.26457,8.59125,9.47584,9.02612,9.43325,9.94526,8.84451
4,10.1440,10.5792,10.2990,9.61472,10.6351,9.88881,10.68370,10.1489,9.70552,10.5044,...,9.58408,8.76610,9.26305,9.16509,8.60445,9.45448,9.04695,9.39320,9.78742,8.76685


In [5]:
#function to compute the correlations
from scipy import stats
def compute_pairwise_correlations(df, start_i=0, start_j=0, end_i=None, end_j=None):
    if end_i is None:
        end_i = len(df.columns)
    if end_j is None:
        end_j = len(df.columns)
    
    cols = df.columns.tolist()
    results = []
    
    for i in range(start_i, end_i):
        for j in range(start_j, end_j):
            if i >= j:  # avoid duplicates and self-correlation
                continue
            col_i = cols[i]
            col_j = cols[j]
            
            # drop rows where either column is null
            #temp contains the two gene columns we are computing correlation for
            temp = df[[col_i, col_j]].dropna()

            #skip over the columns where they are all the same value so no correlation can be computed 
            if temp[col_i].std() == 0 or temp[col_j].std() == 0:
                continue
            
            r, p = stats.pearsonr(temp[col_i], temp[col_j])
            #n is the number of rows that were used to compute r
            n = len(temp)
            
            # t-statistic
            t_stat = r * ((n - 2) ** 0.5) / ((1 - r**2) ** 0.5)
            
            results.append({
                'i':i,
                'j':j,
                'col_i': col_i,
                'col_j': col_j,
                'r': r,
                't_stat': t_stat,
                'p_value': p,
                'n': n
            })
    
    return pd.DataFrame(results)


# all columns
#result_df = compute_pairwise_correlations(df1)

# only specific range
#result_df = compute_pairwise_correlations(df1, start_i=0, start_j=1, end_i=5, end_j=10)

In [7]:
#compute the correlations between all pairs of genes
results_df = compute_pairwise_correlations(df1)
results_df.head(100)

,i,j,col_i,col_j,r,t_stat,p_value,n
0,0,1,gltI,nlpA,0.456745,12.439408,1.075502e-31,589
1,0,2,gltI,purL,0.573864,16.977369,6.908764e-53,589
2,0,3,gltI,nadB,0.544355,15.722219,9.640789e-47,589
3,0,4,gltI,nadA,0.520416,14.765786,3.457903e-42,589
4,0,5,gltI,xanP,0.449160,12.180062,1.382773e-30,589
...,...,...,...,...,...,...,...,...
95,0,96,gltI,yrhB,0.619970,19.143738,7.777899e-64,589
96,0,97,gltI,pheP,0.624273,19.361014,5.947612e-65,589
97,0,98,gltI,livM,0.157926,3.874879,1.186969e-04,589
98,0,99,gltI,ydfE,0.589823,17.696247,1.770891e-56,589


In [8]:
#save as csv or tsv or both
results_df.to_csv('pairwise_correlations.csv', index=False)



In [ ]:
#filter on the genes that only have a highly significant correlation and sig p-value